In [ ]:
!pip install -U langgraph langchain langchain-groq langchain-tavily pygraphviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.16
    Uninstalling langchain-1.3.16:
      Successfully uninstalled langchain-1.3.16


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

In [ ]:
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_tavily import TavilySearch

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [ ]:
import getpass
import os

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n")

Tavily API key:
··········


In [ ]:
tool = TavilySearch(max_results=2)
print(type(tool))
print(tool.name)

<class 'langchain_tavily.tavily_search.TavilySearch'>
tavily_search


In [ ]:
class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
checkpointer = InMemorySaver()
memory = InMemoryStore()

In [ ]:
class Agent:
  def __init__(self,model,tools,checkpointer,system=""):
    self.system=system
    graph=StateGraph(AgentState)
    graph.add_node("llm",self.call_groq)
    graph.add_node("action", self.take_action)
    graph.add_conditional_edges("llm",self.exists_action,{True: "action", False: END})
    graph.add_edge("action", "llm")
    graph.set_entry_point("llm")
    self.graph = graph.compile(checkpointer=checkpointer)
    self.tools = {t.name: t for t in tools}
    self.model= model.bind_tools(tools)

  def call_groq(self, state: AgentState):
    messages = state["messages"]
    if self.system:
      messages = [SystemMessage(content=self.system)]+messages
    message = self.model.invoke(messages)
    return {"messages": [message]}

  def exists_action(self, state:AgentState):
    result = state["messages"][-1]
    return len(result.tool_calls)>0

  def take_action(self, state:AgentState):
    tool_calls = state["messages"][-1].tool_calls
    # print(tool_calls)
    results = []
    for t in tool_calls:
      print(f"calling: {t}")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
    print("Back to the model!")
    return {'messages': results}

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!!
"""

abot = Agent(model, [tool], system=prompt, checkpointer=checkpointer)

In [ ]:
messages = [HumanMessage(content="what is the weather in la")]

In [ ]:
thread = {"configurable":{"thread_id":"1"}}

In [ ]:
# for event in abot.graph.stream({"messages":messages}, thread):
#   for v in event.values():
#     print(v["messages"])

In [ ]:
messages = [HumanMessage(content="What is the weather in sf?")]
result = abot.graph.invoke({"messages": messages}, thread)

calling: {'name': 'tavily_search', 'args': {'query': 'current weather San Francisco', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_f22e78c4-ab69-4498-9ce6-30c3e7d924e3', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'query': 'San Francisco current temperature weather.com', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_db398a5d-20ae-499a-a921-3a367f7cffb6', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'query': 'San Francisco weather now temperature Fahrenheit site:weather.com', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_d8917541-5495-483b-b18f-4b8e88009f35', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'query': 'San Francisco weather now temperature 2024-08-31', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_a20cda6b-55b7-41fd-9d9d-19a73842419c', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result["messages"][-1].content))

**Current weather in San Francisco (as of the latest observation)**  

| Parameter | Value |
|-----------|-------|
| **Temperature** | **≈ 59 °F** (≈ 15 °C) |
| **Condition** | Mostly few clouds (partly‑cloudy) |
| **Wind** | From the NW at about 9 mph (≈ 14 km/h) |
| **Humidity** | ~ 81 % |
| **Pressure** | ~ 1012 hPa |
| **Sunrise / Sunset** | 06:40 am / 07:42 pm (local time) |

*Source:* i‑weather.com’s “Hourly weather today – Aug 30” page for San Francisco, which lists the current observation as “Now – 59 °F, few clouds, wind 228° 9 mph, 81 % humidity, 1012.1 mb pressure”【2†results.0.content】.  

So, right now San Francisco is cool and partly cloudy with temperatures in the mid‑50s °F, light north‑westerly winds, and relatively high humidity.

In [ ]:
messages = [HumanMessage(content= "what about NewYork")]
result = abot.graph.invoke({"messages":messages}, thread)

calling: {'name': 'tavily_search', 'args': {'query': 'New York current temperature weather now', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_cd4e9735-2da3-457e-9676-79394aef314a', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'query': 'New York hourly weather today i-weather.com', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_8b0c679f-69f0-47b3-bd28-d2441f5709c9', 'type': 'tool_call'}
Back to the model!
calling: {'name': 'tavily_search', 'args': {'query': 'New York current temperature weather.gov', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_f3c3a8b7-c3c4-4ce7-b4aa-76f241c7b41f', 'type': 'tool_call'}
Back to the model!


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01jwat3393fx1937sxcza1x4j0` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8679, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}